# Magnetic Materials

A permanent magnet, soft iron and a conductor occupy the same domain and share one solved magnetic field. A coil pulse disturbs that field: the iron changes its magnetization, while induced currents delay the conductor's response.

The field is modelled as a spacetime bivector, and magnetic moments as intrinsic patches that pair with it. The permanent magnet carries a prescribed moment. In the iron, a material map relates the moment to the local field. The pairing supplies the magnetic coupling in both cases.

In [ ]:
from functools import reduce
from operator import and_
import numpy as np
from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.electromagnetism.magnetic_materials import scenarios, render
from examples.util.animation import save_figure, show_figure, show_animation

ga = Algebra("x+y+z+t-w0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                       # Scalar+
Point = ga.gatype.antivector()                    # Antivector-
Field = ga.gatype.bivector()                      # Bivector+
Moment = ga.gatype.antibivector()                 # Antibivector-
origin = mv.w.dual()
time = -mv.t.dual()
unread = mv.z.dual() & time
patches = stack((mv.xz.dual(), mv.yz.dual()))
magnetic_response = (patches * (Field & patches)).sum(axis=0)
sweep = mv.z.dual() & time
electric_response = sweep * (Field & sweep)


## Materials in one domain

The permanently magnetized bar supplies a field, the surrounding iron concentrates its flux, and the copper responds to changes by carrying current. Their material properties vary across resolved transitions on the shared mesh. The two coil cross-sections carry equal and opposite prescribed currents.

The model is uniform along z. Currents run into and out of the picture; magnetic field lines lie within it. The iron's induced moment follows the local field reversibly and vanishes when that field vanishes.

In [ ]:
n_across, n_up, n_frames = scenarios.n_across, scenarios.n_up, scenarios.n_frames
frame_step = scenarios.frame_step
peak_drive = scenarios.coil_current
topology, vertices, fractions, permeability, conductivity, remanence, coils, drive = scenarios.scene(n_across, n_up, n_frames)
figure = render.setup(topology, vertices, fractions, coils, peak_drive)
save_figure(figure, "magnetic_materials_setup")
show_figure(figure)

## The magnetic field

The spatial potential is `potential * z`. Its exterior derivative gives the magnetic field. In this cross-section, curves of constant `potential` follow the magnetic field lines.

For these linear materials, `field & response(field) / 2` is the magnetic storage term. The permanent magnet's contribution lowers the energy when the field aligns with its prescribed moment. The moment density and permeability are material fields; changes at their edges enter the same field equation as their interiors.

Only the remote boundary holds the potential fixed. The field is solved inside every material, including the permanent magnet.

In [ ]:
corners = topology.incidence[2, 0]
mean = corners.mean(axis=0)
spans = reduce(and_, corners * vertices, unread)
areas = spans.norm() / 2
cells = ~mean * areas
facing = topology.off[2, 0] * vertices
complements = facing[0] & facing[1] & unread
hats = complements / (complements & (corners * vertices))
gradients = hats - (hats & origin) * mv.w
# A spatial potential along z gives the magnetic field within this cross-section.
hat_fields = gradients ^ mv.z
response = magnetic_response / (mean * permeability)
local = areas * (hat_fields[:, None] & response(hat_fields[None, :]))
stiffness = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)
source = (~corners * (areas * (hat_fields & (mean * remanence)))).sum(axis=0)
mass = cells * conductivity
extension = ~topology.reading(topology.interior[0], 0)

coil_shape = coils / (cells * coils).sites.sum()
coil = cells * (coil_shape * scenarios.coil_signs).sum(axis=0)

## Induced currents

A changing spatial potential drives an electric field along z. Conductivity turns that field into current, whose magnetic field opposes the change. The midpoint step couples magnetic storage to this resistive response.

The isolated conductor has no terminals supplying a net current along z. Charge redistribution supplies a uniform electric field in that direction so currents into and out of the cross-section balance. Opposite currents still flow through different parts of the conductor. This charge redistribution is treated as fast compared with the magnetic relaxation followed here.

In [ ]:
def evolution(stiffness, source: Scalar, mass: Scalar, extension, coil: Scalar,
              drive: np.ndarray, step: float):
    """Evolve one spatial potential under a coil pulse, with the magnet always present."""
    initial = (~extension * stiffness * extension) * Scalar
    potential = extension * initial.solve(~extension * (source + coil * drive[0]))
    yield potential
    storage = spdiag(mass / step)
    implicit = (~extension * (storage + stiffness / 2) * extension) * Scalar
    explicit = storage - stiffness / 2
    # An isolated conductor redistributes charge until its net current along z vanishes.
    # Its uniform electric reading is determined alongside the spatial potential.
    uniform = extension * implicit.solve(~extension * (mass / step))
    total_mass = mass.sites.sum()
    denominator = 1 - (mass * uniform).sites.sum() / total_mass
    for previous, following in zip(drive[:-1], drive[1:]):
        load = source + coil * ((previous + following) / 2)
        mean_potential = (mass * potential).sites.sum() / total_mass
        particular = extension * implicit.solve(~extension * (explicit * potential + load - mass * mean_potential / step))
        mean_following = (mass * particular).sites.sum() / total_mass / denominator
        potential = particular + uniform * mean_following
        yield potential

## Magnetic poles

The uniformly magnetized bar has opposite effective poles at its ends. In the iron, the induced magnetization produces poles whose distribution changes with the coil pulse. The pole density is read from the spatial variation of the total magnetization, so its surface layers follow the material transitions already present on the mesh.

These equivalent charges describe magnetization; they are not free magnetic monopoles. The magnetic flux remains continuous through the materials.

In [ ]:
# The spatial volume supplies the magnetic plane that reads each boundary moment.
pole_readings = gradients | mv.xyz
permanent = mean * remanence

## A coil pulse

The permanent magnet remains present throughout the pulse. The coil first opposes part of its field, then switches off. Induced currents continue to change the field after the prescribed current has ended. The next cell takes a few seconds to compute and draw the animation.

In [ ]:
potentials = []

def observed():
    previous = None
    for index, potential in enumerate(evolution(stiffness, source, mass, extension, coil, drive, frame_step)):
        rate = 0 * potential if previous is None else (potential - previous) / frame_step
        field = (hat_fields * (corners * potential)).sum(axis=0)
        magnetization = permanent + magnetic_response(field) - response(field)
        # Moment flux through each cell's boundary gives its equivalent magnetic pole charge.
        pole_charge = (~corners * (areas * (pole_readings & magnetization))).sum(axis=0)
        pole_density = pole_charge / cells
        mean_rate = (mass * rate).sites.sum() / mass.sites.sum()
        electric = -(rate - mean_rate) * mv.zt
        current = conductivity * electric_response(electric)
        potentials.append(potential)
        previous = potential
        yield potential, pole_density, current, drive[index]

show_animation(render.animate(topology, vertices, fractions, coils, peak_drive, observed()),
               "magnetic_materials_notebook", 40, n_frames=n_frames)

The gold curves are magnetic field lines. Red and blue show positive and negative effective magnetic pole density, on the same scale in the permanent magnet and the iron. Higher densities become lighter. Green and violet show opposite electric currents perpendicular to the cross-section, in both the copper and the prescribed coil.

## Dissipation

The conductor continues to dissipate energy as its induced currents relax. The dotted line marks the end of the coil pulse.

In [ ]:
history = stack(potentials)
figure = render.relaxation(history, mass, frame_step, scenarios.pulse_fall)
save_figure(figure, "magnetic_materials_notebook_relaxation")
show_figure(figure)

In [ ]:
# checks
assert np.all(history.isfinite())
rate = (history[1:] - history[:-1]) / frame_step
rate = rate - (mass * rate).sites.sum() / mass.sites.sum()
assert np.all((mass * rate).sites.sum().abs() < .01)